# Tiny Turn Detector - Training in Google Colab

This notebook trains a turn detection model using the pipecat-ai dataset.

**Setup:**
1. Go to `Runtime` → `Change runtime type` → Select **GPU** (T4)
2. Run all cells in order

---

## Step 1: Upload Your Project ZIP

In [ ]:
from google.colab import files
import os

print("Please upload your tiny-turn-detector.zip file:")
uploaded = files.upload()

# Get the filename
zip_filename = list(uploaded.keys())[0]
print(f"\nUploaded: {zip_filename}")

## Step 2: Extract and Setup Project

In [ ]:
!unzip -q {zip_filename}
%cd tiny-turn-detector

print("\n=== Project Structure ===")
!ls -la

## Step 3: Install Dependencies

In [ ]:
print("Installing dependencies...")
!pip install -q torch torchaudio transformers datasets librosa soundfile \
              scikit-learn numpy pandas pyyaml tqdm

print("\n✓ Dependencies installed!")

## Step 4: Check GPU Availability

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Update config to use GPU
    print("\nUpdating config to use GPU...")
    !sed -i 's/device: "cpu"/device: "cuda"/' configs/config.yaml
    !sed -i 's/device: cpu/device: cuda/' configs/config.yaml
    print("✓ Config updated to use GPU")
else:
    print("\n⚠️ GPU not available. Training will use CPU (slower).")
    print("To enable GPU: Runtime → Change runtime type → GPU")

## Step 5: Prepare Dataset

This will:
- Download the audio dataset from HuggingFace
- Create train/validation/test splits
- Takes ~1-2 minutes

In [ ]:
!python data/prepare.py

## Step 6: Train the Model

This will train for up to 10 epochs with early stopping.

**Expected time:**
- With GPU: ~5-10 minutes
- With CPU: ~30-60 minutes

In [ ]:
!python train.py

## Step 7: View Training Results

In [ ]:
print("=== Trained Model ===")
!ls -lh checkpoints/

import os
if os.path.exists('checkpoints/best_model.pt'):
    size = os.path.getsize('checkpoints/best_model.pt') / (1024 * 1024)
    print(f"\n✓ Model saved: best_model.pt ({size:.2f} MB)")
else:
    print("\n⚠️ Model checkpoint not found")

## Step 8: Download Trained Model

In [ ]:
from google.colab import files

print("Downloading trained model...")
files.download('checkpoints/best_model.pt')

print("\n✓ Model downloaded! Check your browser's download folder.")

## Step 9: Test Inference (Optional)

Upload an audio file to test the trained model.

In [ ]:
from google.colab import files

print("Upload an audio file (.wav, .mp3, etc.):")
test_audio = files.upload()

if test_audio:
    audio_filename = list(test_audio.keys())[0]
    print(f"\nRunning inference on: {audio_filename}")
    !python inference.py {audio_filename}
else:
    print("No file uploaded.")

## Optional: Save to Google Drive

Save your trained model to Google Drive for permanent storage.

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create directory and copy model
!mkdir -p '/content/drive/MyDrive/TinyTurnDetector'
!cp -r checkpoints '/content/drive/MyDrive/TinyTurnDetector/'
!cp configs/config.yaml '/content/drive/MyDrive/TinyTurnDetector/'

print("\n✓ Model saved to Google Drive: MyDrive/TinyTurnDetector/")